# Title: Housing Price Prediction
## Overview of the Project
In this notebook, I build a machine learning project that predicts the price of houses as influenced by different features including the median_income, the proximity to the ocean, the number of rooms among others.
The goal is to found out how the different features relate to the price of houses in California and create a model that would accurately predict the prices of houses.


# Libraries
Let's first import the relevant libraries

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#preprocessing libraries
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#ml algorithms
from sklearn.linear_model import LinearRegression

# Dataset
Let's import the data. I used California house price datasets that can easily be imported from kaggle.

In [ ]:
df = pd.read_csv('california.csv')

# Simple glimpse at the data

In [ ]:
df.head()

In [ ]:
df.shape

# Features
From the glimpse, the dataset is made pf 20640 rows and 10 columns. TThe columns can be divided into different categories. there are numerical and non-numerical data. There are geographical and human features. Geographical features include the longitutde and latitude as well as ocea_proximity. Demographic Features include the population and households and housing_median_age. Structural features include total_rooms, total bedrooms. Financial features include the median_house_value and median income. The target is the median_house_value. THis notebook is set to predict the median_house_price as influenced by the different features presented

First let's check the range of the median_housing_value

# QUick EDA and data encoding
This sections gives a visual overview of the data, the range, the relationship between the data. The first is the boxplot which gives a picture of the range of the target feature, median_house_value

In [ ]:
sns.boxplot(df['median_house_value'])

# insights
From the plot, it is evident that the median price of houses is about 200000, and falls between 100000 and 300000. There are not many outliers

Next We establish how each of the features relate with the target variable. This will be shown using a heatmap

# Encoding and heatmap
One of the features of the dataset is ocean proximity which is made of categorical data. The next EDA is to establish the relationship between the different features and the target. In this I will use a correlation heatmap. This approach does not work on categorical data hence the need for conversion from categorical to numerical data using the get_dummies() method.


In [ ]:
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
corre = df.corr()
sns.heatmap(corre, annot = True, cmap='coolwarm')

the correlation heatmap shows some features like the median income having a high corelation to the median house value, 0.69, while some like the population have the lowest at 0.025.

In [ ]:
sns.pairplot(df)

# Missing values
One of the preprocessing activities  had to also deal with was checking and handling missing values since having them in the dataset would affect the accuracy of the model.

In [ ]:
df.isnull().sum()

Only one column (total_bedrooms)has missing values. there are 207 rows of missing values in thus. checked against the total number of rows 20640, that is 1 percent of the rows. removing this rows is less likely to affect the outcome of the model. Alternatively, I could fill the missing values with either a 0, the mean, the median or modal value. I chose to fill the missing rows with the mean.

In [ ]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].mean())

# Features
Setting the X and y features

In [ ]:
X = df.drop(['median_house_value'], axis=1)
y = df['median_house_value']

Split the data
Next was to split the data into train and test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10, shuffle=True)

# Standardization
The data in this dataset had different ranges and this would make it difficult for the model to give an accurate prediction and hence need for standardization

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Time to intoduce the algorithm

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(X_train, y_train)

Let's get the model intercepts and coefficients to use in the prediction

In [ ]:
model.intercept_

In [ ]:
model.coef_

Let's organize the coefficients into a table

In [ ]:
model_results = pd.DataFrame(model.coef_, X.columns, columns = ['coefficients'])
model_results

In [ ]:
new_data = [[
    -150, #longitude
    40, # latirude
    40, #housingm_median_age
    5, # total_rooms
    3, #total_bedrooms
    2500, #population
    1000, #households
    9, #median_income
    0, #inland
    0, #island
    1, #nearbay
    0, #near_ocean
]]
new_data_df = pd.DataFrame(new_data, columns=X.columns)
new_data_scaled = scaler.transform(new_data_df)
model_prediction = model.predict(new_data_scaled)
model_prediction

# Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
y_pred = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, y_pred))
print('R2_score:', r2_score(y_test, y_pred))
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('RMSE:', rmse)

# Metrics
The mean_absolute error shows that the predicted price is off by 50050. The r2_score is 0.64, which points out that 64 percent of the changes in the median_house_value is affected by the features highlighted in the dataset while 36 percent of the changes are caused by features not indicated in the dataset.

# Model Improvement

Initially, the model produced a very high RMSE due to preprocessing issues like standardization and data encoding.

After correcting this, (standardization and feature encoding, I had initially removed the column with non_numerical data), the RMSE improved significantly to 69,132, indicating a much better model fit.
##  Model Evaluation

An RMSE of 69,132 means that, on average, the model's predictions deviate from the actual house prices by about 69,132.

This level of error is reasonable given the price range in the dataset.

# Conclusion

- The model successfully learns patterns in housing data
- Proper preprocessing was critical to performance
- Further improvements. I plan to expore other models like RF and XGboost to see which model has a better performance